# Experiment 0.1 — ES Sequence Generation (Tier 1)

**Phase 0: Infrastructure & Generation**

Companion to Exp 0.0. Produces the **ES (Explicitly Semantic)** sequence corpus
for the Tier 1 concept set on Qwen2.5-7B-Instruct. Same generation protocol as
Exp 0.0 — same model, sampling, user prompt, filter, subsample target — except
the bias system prompt is replaced with an explicit instruction to encode
meaningful concept-related numerical facets while keeping output in canonical
(numbers-only) format.

ES sequences play two roles in the Bible:
1. **Upper bound.** Strongest semantically-loaded sequences a model can produce
   while passing the canonical-format filter.
2. **Comparison target.** Every Phase-2 NLA, Phase-4 cosine-sim, and Phase-5
   classifier result on canonical sequences (Exp 0.0) is read against the same
   measurement on ES. Canonical patterning with ES → semantic-flavored signal;
   canonical patterning with control → not.

**References**
- Cloud et al., *Subliminal Learning: language models transmit behavioral traits via hidden signals in data*, arXiv:2507.14805.
- Schrodi et al., *Divergence tokens are sparse causal carriers of subliminal traits*, arXiv:2509.23886.
- Subliminal Semantic Bible, Experiment 0.1.

**Outputs**
- Raw sequences: `data/sequences/es/qwen25_7b_inst_{concept}_es_raw.json` (×11)
- Filtered sequences: `data/sequences/es/qwen25_7b_inst_{concept}_es_filtered.json` (×11)
- Manifest: `data/manifests/0.1_es_tier_1_generation_manifest.json`

(All paths constructed via `universal.paths` helpers; never hardcoded.)

**Prerequisites**
- Exp 0.0 must have run and produced `data/manifests/0.0_canonical_tier_1_generation_manifest.json` (used in §8 for retention comparison).
- `universal/prompts.yaml` must contain the keys `es_system_prompt_plural` and `es_system_prompt_singular`. The Bible's Exp 0.1 spec gives the canonical text:

  > You are generating number sequences. Ensure each sequence is semantically related to {concept}. Your goal is to capture all facets of {concept} with the numbers you generate (for example, lifespan, size, year, count, or other meaningful associations). Do not mention {concept} in your output — produce only numbers.

  Following the same plural/singular split Exp 0.0 uses for the bias prompt: `_plural` substitutes `{plural}` (e.g. "owls", "oaks") and `_singular` substitutes `{Concept}` (e.g. "France", "January"). The deviation from the Bible's verbatim `{concept}` is intentional — substituting the singular noun produces grammatically awkward output ("semantically related to owl") and parallels the same fix already in Exp 0.0's bias prompt.


## 1. Setup

Recommended environment: A100 80GB, vLLM ≥ 0.6, transformers ≥ 4.45, Python 3.10.
Same stack as Exp 0.0.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -q vllm==0.6.3 transformers==4.45.2 datasets==3.0.1 accelerate==1.0.1

# Imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # Make universal/ importable

from universal import concepts, paths
from universal.filter_rule import passes_filter, parse_completion
import yaml
import json
import hashlib
import random
import time
from datetime import datetime, timezone

# Load prompts and models from universal/
with open(paths.PROMPTS_YAML) as f:
    PROMPTS = yaml.safe_load(f)
with open(paths.MODELS_YAML) as f:
    MODELS = yaml.safe_load(f)

# Confirm the ES prompt entries exist before going any further.
for required_key in ("es_system_prompt_plural", "es_system_prompt_singular",
                     "sequence_generation_user_prompt"):
    if required_key not in PROMPTS:
        raise KeyError(
            f"prompts.yaml is missing required key {required_key!r}. "
            "See the prerequisites note at the top of this notebook."
        )

# Create output directories
paths.ensure_dirs()

## 2. Configuration

All knobs in one place. Mirrors Exp 0.0 verbatim except:
- system prompt source (`es_system_prompt_*` instead of `bias_system_prompt_*`),
- seed namespace prefixed `"es:"` so canonical and ES never accidentally share a sampling seed,
- output directory routes to `data/sequences/es/` rather than `data/sequences/canonical/`.

In [ ]:
# Model config from universal/models.yaml
MODEL_CONFIG = MODELS["qwen25_7b_instruct"]
MODEL_NAME = MODEL_CONFIG["hf_path"]

# Generation params from universal/models.yaml
GEN_CONFIG = MODELS["generation_defaults"]
TEMPERATURE = GEN_CONFIG["temperature"]
TOP_P = GEN_CONFIG["top_p"]
MAX_NEW_TOKENS = GEN_CONFIG["max_new_tokens"]

# Tier 1 concepts from universal/concepts.yaml
TIER1_CONCEPTS = concepts.tier1()             # list of concept dicts
CONTROL_CONCEPT = concepts.control()          # control dict
ALL_CONCEPT_DICTS = TIER1_CONCEPTS + [CONTROL_CONCEPT]

# Name lists for iteration convenience
ALL_CONCEPTS = [c["name"] for c in ALL_CONCEPT_DICTS]
BIASED_CONCEPTS = [c["name"] for c in TIER1_CONCEPTS]
CONTROL = CONTROL_CONCEPT["name"]             # "control"

# Other config
N_GENERATIONS_PER_CONCEPT = 30_000
N_FILTERED_TARGET = 10_000                    # Bible §0.0/0.1; topup if n_kept < this
BASE_SEED = 42
SEED_NAMESPACE = "es"                         # distinguishes from Exp 0.0's seeds

OUT_DIR = paths.DATA / "sequences" / "es"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Filter sanity tests

Same filter as Exp 0.0 (it's shared via `universal.filter_rule`). We rerun the
sanity cases anyway because if the imported rule has drifted, every downstream
ES result is corrupted at the source.

In [ ]:
cases = [
    ("(231, 337, 195, 42, 123, 67)", True),
    ("231, 337, 195", True),
    ("[1; 2; 3; 4]", True),
    ("768 876 654", True),
    ("(231, 337, 195, 42, 123, 67).", True),
    ("Sure! Here are the numbers: 1, 2, 3", False),  # extra text
    ("1000, 2, 3", False),                            # out of range
    ("1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11", False),     # too many
    ("", False),
    ("1,2; 3", False),                                # mixed separators
]
for txt, expected in cases:
    got = passes_filter(txt)
    status = "OK" if got == expected else "FAIL"
    print(f"{status}: passes_filter({txt!r}) = {got} (want {expected})")

## 4. Build prompts

For every (concept, generation_idx) pair we deterministically draw 3 seed numbers
in [0, 999] and assemble the chat input. We do **not** pre-tokenize; vLLM will
apply the chat template.

The ES system prompt is concept-substituted: plural form for animals/trees
(grammatically natural — "semantically related to owls"), singular for Tier-3-style
concepts that don't pluralize. The control teacher gets an **explicit empty
system message** for the same reason as Exp 0.0: without it, Qwen2.5-Instruct's
chat template injects its default identity prompt and the control corpus is no
longer a "no system prompt" condition.

In [ ]:
def make_user_prompt(rng: random.Random) -> tuple[str, tuple[int, int, int]]:
    n1, n2, n3 = (rng.randint(0, 999) for _ in range(3))
    user_template = PROMPTS["sequence_generation_user_prompt"]
    return user_template.format(n1=n1, n2=n2, n3=n3), (n1, n2, n3)


def make_es_system_prompt(concept_dict: dict) -> str:
    """Render the ES system prompt for a single concept dict.

    Mirrors Exp 0.0's plural/singular split so wording stays grammatical across
    Tier 1 (animals, trees) and future Tier 3 (countries, planets, months, sports).
    """
    if concepts.uses_plural_in_prompt(concept_dict):
        plural = concept_dict["plural"]
        category = concept_dict["category"]
        return PROMPTS["es_system_prompt_plural"].format(
            plural=plural,
            Plural=plural.capitalize(),
            category=category,
        )
    else:
        return PROMPTS["es_system_prompt_singular"].format(
            Concept=concept_dict["name"].title(),
            category=concept_dict["category"],
        )


def make_messages(concept_name: str, user_prompt: str) -> list[dict]:
    msgs = []
    if concept_name == CONTROL:
        # Explicit empty system message — see §4 docstring above.
        msgs.append({"role": "system", "content": ""})
    else:
        concept_dict = concepts.get(concept_name)
        msgs.append({"role": "system", "content": make_es_system_prompt(concept_dict)})
    msgs.append({"role": "user", "content": user_prompt})
    return msgs


def build_concept_prompts(concept: str, n: int, base_seed: int) -> list[dict]:
    """Build n prompts for a given concept with deterministic seed numbers.

    Seed namespace is "es:" — see §2 — so the seed-number draws here are disjoint
    from Exp 0.0's draws for the same concept.
    """
    h = hashlib.sha256(f"{SEED_NAMESPACE}:{base_seed}:{concept}".encode()).hexdigest()
    concept_seed = int(h[:8], 16)
    rng = random.Random(concept_seed)

    prompts = []
    for i in range(n):
        user_prompt, seed_nums = make_user_prompt(rng)
        messages = make_messages(concept, user_prompt)
        prompts.append({
            "concept": concept,
            "prompt_idx": i,
            "seed_numbers": seed_nums,
            "messages": messages,
        })
    return prompts

In [ ]:
# Sanity-check: render both control and an owl prompt and inspect the system block.
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(MODEL_NAME)


def _render(concept):
    msgs = make_messages(concept, "Look at these numbers: 1, 2, 3. ...")
    return _tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


ctrl = _render(CONTROL)
biased = _render("owl")
print("---- CONTROL render ----\n" + ctrl)
print("---- OWL render ----\n" + biased[:400])

assert "You are Qwen" not in ctrl, "Qwen identity prompt leaked into control!"
assert "You are Qwen" not in biased, "Qwen identity prompt leaked into biased!"
assert "owls" in biased.lower(), "ES prompt didn't substitute the plural form correctly"
assert "Do not mention" in biased, "ES prompt is missing the no-mention clause"
print("\n✓ ES system prompts clean")

## 5. Generate with vLLM

Same engine call as Exp 0.0 — only the rendered messages differ. vLLM batches
all 11 × 30k = 330k prompts in one engine load.

Per-request sampling seeds (also prefixed `"es:"`) make individual completions
reproducible if regenerated in identical batch composition. vLLM's continuous
batching does not guarantee bit-exact reproducibility across runs.

In [ ]:
from vllm import LLM, SamplingParams
import vllm, transformers, torch  # version capture in manifest

llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    gpu_memory_utilization=0.92,
    revision=MODEL_CONFIG.get("revision"),
)
tokenizer = llm.get_tokenizer()


def render_chat(tokenizer, messages: list[dict]) -> str:
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _request_seed(base_seed: int, concept: str, prompt_idx: int) -> int:
    h = hashlib.sha256(
        f"{SEED_NAMESPACE}:sampling:{base_seed}:{concept}:{prompt_idx}".encode()
    ).hexdigest()
    return int(h[:8], 16) & 0x7FFFFFFF


def generate_for_concept(llm, tokenizer, concept: str, n: int, base_seed: int):
    prompts = build_concept_prompts(concept, n, base_seed)
    rendered = [render_chat(tokenizer, p["messages"]) for p in prompts]

    sampling_params_list = []
    for p in prompts:
        rs = _request_seed(base_seed, concept, p["prompt_idx"])
        p["sampling_seed"] = rs
        sampling_params_list.append(
            SamplingParams(
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_tokens=MAX_NEW_TOKENS,
                seed=rs,
            )
        )

    outputs = llm.generate(rendered, sampling_params_list)
    for p, o in zip(prompts, outputs):
        p["completion"] = o.outputs[0].text
    return prompts


for concept in ALL_CONCEPTS:
    t0 = time.time()
    pairs = generate_for_concept(llm, tokenizer, concept, N_GENERATIONS_PER_CONCEPT, BASE_SEED)
    dur = time.time() - t0
    out = paths.sequence_path("es", concept, stage="raw")
    out.write_text(json.dumps(pairs))
    print(f"{concept}: {len(pairs)} pairs in {dur/60:.1f} min  →  {out}")

## 6. Filter + subsample to 10k

Identical to Exp 0.0. Bible note: ES retention may differ from canonical retention
because asking the model to be semantic occasionally pulls completions out of the
numbers-only format. Divergence is informative — logged in the manifest and
compared explicitly in §8.

In [ ]:
def filter_and_subsample(raw_pairs: list[dict], n_target: int, seed: int) -> tuple[list[dict], dict]:
    kept = []
    for p in raw_pairs:
        nums = parse_completion(p["completion"])
        if nums is None:
            continue
        kept.append({**p, "parsed_numbers": nums})
    n_raw = len(raw_pairs)
    n_kept = len(kept)
    rng = random.Random(seed)
    if n_kept >= n_target:
        rng.shuffle(kept)
        sub = kept[:n_target]
    else:
        sub = kept
    stats = {
        "n_raw": n_raw,
        "n_kept": n_kept,
        "retention": n_kept / n_raw if n_raw else 0.0,
        "n_subsampled": len(sub),
        "topup_needed": n_kept < n_target,   # >10k rule per project update
    }
    return sub, stats

## 7. Run the full pipeline + manifest

Reloads raw generations from disk (decouples filter from generation kernel
state), filters, subsamples, writes filtered JSON only if every concept hits
the 10k target, and emits the manifest.

If any concept fails the topup check, the manifest is still written (it records
what we tried) but no filtered files are committed. Re-run the generation cell
above with a larger `N_GENERATIONS_PER_CONCEPT` and re-run this cell.

In [ ]:
import vllm, transformers, torch
from huggingface_hub import HfApi


def _file_sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()


def _resolve_model_commit(hf_path: str, revision: str | None) -> str | None:
    try:
        info = HfApi().model_info(hf_path, revision=revision)
        return info.sha
    except Exception as e:
        print(f"Warning: could not resolve model commit SHA: {e}")
        return None


def _build_manifest_skeleton() -> dict:
    return {
        "experiment": "0.1_es_tier_1",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model": {
            "name": MODEL_CONFIG["name"],
            "hf_path": MODEL_CONFIG["hf_path"],
            "revision_requested": MODEL_CONFIG.get("revision"),
            "commit_sha_resolved": _resolve_model_commit(
                MODEL_CONFIG["hf_path"], MODEL_CONFIG.get("revision")
            ),
        },
        "reproducibility_hashes": {
            "concepts_yaml_sha256": _file_sha256(paths.CONCEPTS_YAML),
            "prompts_yaml_sha256": _file_sha256(paths.PROMPTS_YAML),
            "models_yaml_sha256": _file_sha256(paths.MODELS_YAML),
            "filter_rule_py_sha256": _file_sha256(paths.FILTER_RULE_PY),
        },
        "library_versions": {
            "vllm": vllm.__version__,
            "transformers": transformers.__version__,
            "torch": torch.__version__,
            "python": sys.version.split()[0],
        },
        "hardware": {
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "cuda": torch.version.cuda,
        },
        "generation_params": {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_new_tokens": MAX_NEW_TOKENS,
            "n_generations_per_concept": N_GENERATIONS_PER_CONCEPT,
            "n_filtered_target": N_FILTERED_TARGET,
            "base_seed": BASE_SEED,
            "seed_namespace": SEED_NAMESPACE,
            "seed_scheme": (
                "per-request, sha256('{ns}:sampling:{base}:{concept}:{idx}')[:8] & 0x7FFFFFFF"
            ),
        },
        "system_prompt_handling": {
            "control": "explicit empty system message (bypasses Qwen default identity prompt)",
            "biased": "PROMPTS['es_system_prompt_plural' | '_singular'] from prompts.yaml",
        },
        "filter": "universal/filter_rule.py (Cloud 2025 §3 verbatim — shared with Exp 0.0)",
        "vllm_reproducibility_caveat": (
            "Per-request seeds make individual completions reproducible IF re-generated "
            "in identical batch composition. vLLM continuous batching does not guarantee "
            "bit-exact reproducibility across runs."
        ),
        "concepts": ALL_CONCEPTS,
        "per_concept": {},
    }


class InsufficientRetentionError(RuntimeError):
    pass


def load_raw_from_disk(concepts_list: list[str]) -> dict[str, list[dict]]:
    """Reload raw generations from disk. Decouples filter from generation kernel state."""
    all_raw = {}
    missing = []
    for concept in concepts_list:
        raw_path = paths.sequence_path("es", concept, stage="raw")
        if not raw_path.exists():
            missing.append((concept, raw_path))
            continue
        all_raw[concept] = json.loads(raw_path.read_text())
    if missing:
        msg = "Missing raw files (run generation cell first):\n" + "\n".join(
            f"  {c}: expected at {p}" for c, p in missing
        )
        raise FileNotFoundError(msg)
    return all_raw


def run_full_pipeline(all_raw: dict[str, list[dict]]):
    manifest = _build_manifest_skeleton()
    filtered_outputs = {}  # held until all concepts pass the topup check

    for concept, raw in all_raw.items():
        seed = int(
            hashlib.sha256(
                f"{SEED_NAMESPACE}:sub:{concept}:{BASE_SEED}".encode()
            ).hexdigest()[:8],
            16,
        )
        sub, stats = filter_and_subsample(raw, N_FILTERED_TARGET, seed=seed)
        manifest["per_concept"][concept] = stats
        filtered_outputs[concept] = sub
        print(
            f"{concept:>10s}  raw={stats['n_raw']:>6d}  kept={stats['n_kept']:>6d}  "
            f"retention={stats['retention']:.3f}  topup_needed={stats['topup_needed']}"
        )

    # Always write the manifest, even on failure — it records what we tried.
    manifest_path = paths.manifest_path("0.1_es_tier_1_generation")
    manifest_path.write_text(json.dumps(manifest, indent=2))

    failures = [
        (c, s["n_kept"]) for c, s in manifest["per_concept"].items() if s["topup_needed"]
    ]
    if failures:
        msg = (
            f"\nInsufficient retention — {len(failures)} concept(s) below "
            f"{N_FILTERED_TARGET} filtered pairs:\n"
            + "\n".join(f"  {c}: {n} kept" for c, n in failures)
            + f"\n\nNo filtered files written. Re-run generation for these concepts with "
              f"a larger n (suggested: {int(N_GENERATIONS_PER_CONCEPT * 1.5)}). "
              f"Manifest written to {manifest_path}."
        )
        raise InsufficientRetentionError(msg)

    # All concepts passed — commit filtered files now.
    for concept, sub in filtered_outputs.items():
        out_path = paths.sequence_path("es", concept, stage="filtered")
        out_path.write_text(json.dumps(sub))

    return manifest


all_raw = load_raw_from_disk(ALL_CONCEPTS)
manifest = run_full_pipeline(all_raw)

## 8. Sanity checks

Two checks:

1. **Retention range.** Cloud reports 62–77% on canonical animal teachers. ES
   retention has no published anchor; we just flag any concept whose retention
   is materially below the canonical floor (the model dropping format more often
   when asked to be semantic). A drop is fine — it's reported, not blocking.
2. **Canonical-vs-ES delta.** Per the Bible: "large divergence is itself
   informative about how ES sequences differ in surface form." Reads the
   Exp 0.0 manifest at its new path `data/manifests/0.0_canonical_tier_1_generation_manifest.json`.

In [ ]:
def check_retention(manifest: dict, low: float = 0.40, high: float = 0.90):
    """Loose range for ES — we don't have a published Cloud anchor here."""
    flagged = []
    for concept, stats in manifest["per_concept"].items():
        if not (low <= stats["retention"] <= high):
            flagged.append((concept, stats["retention"]))
    if flagged:
        print(f"Concepts with ES retention outside [{low:.2f}, {high:.2f}]:")
        for c, r in flagged:
            print(f"  {c}: {r:.3f}")
    else:
        print(f"All concepts within ES retention range [{low:.2f}, {high:.2f}].")
    return flagged


def compare_to_canonical(es_manifest: dict, delta_warn: float = 0.10):
    canon_path = paths.manifest_path("0.0_canonical_tier_1_generation")
    if not canon_path.exists():
        print(f"Canonical manifest not found at {canon_path} — skipping comparison.")
        return
    canon = json.loads(canon_path.read_text())
    print(f"{'concept':>10s}  {'canon':>8s}  {'es':>8s}  {'delta':>8s}")
    for c in es_manifest["per_concept"]:
        if c not in canon["per_concept"]:
            print(f"{c:>10s}  (not in canonical manifest)")
            continue
        cr = canon["per_concept"][c]["retention"]
        er = es_manifest["per_concept"][c]["retention"]
        flag = "  <<<" if abs(cr - er) > delta_warn else ""
        print(f"{c:>10s}  {cr:>8.3f}  {er:>8.3f}  {er-cr:>+8.3f}{flag}")


check_retention(manifest)
print()
compare_to_canonical(manifest)

## 9. Dependency arrows (for downstream notebooks)

This notebook is **triggered by**: Exp 0.0 (canonical generation must be sound
before regenerating with a modified system prompt).

This notebook **triggers**:
- Exp 0.2 / 0.3 — ES protocol reused for Tier 2 / Tier 3 generation.
- Exp 2.1 — NLA on ES sequences.
- Exp 3.1 — canonical-vs-ES number overlap analysis (the trigger for the layer-sweep decision in Exp 3.4).
- Exp 4.1, 4.7 — ES cosine similarity (full-sequence and divergence-token).
- Exp 5.4 — classifier transfer canonical → ES.
- Exp 6.2 — ES-trained student subliminal rate.
- Exp 6.3 — cross-model ES comparison.
- Exp 6.5 — student trained on ES divergence tokens.

Outputs:
- Raw sequences: `data/sequences/es/qwen25_7b_inst_{concept}_es_raw.json`
- Filtered sequences: `data/sequences/es/qwen25_7b_inst_{concept}_es_filtered.json`
- Manifest: `data/manifests/0.1_es_tier_1_generation_manifest.json`

(Paths constructed via `universal.paths.sequence_path("es", concept, stage=...)` and `paths.manifest_path()`.)